# Generador de Datos Sintéticos V5

## Descripción General

**Propósito**: Generar datasets sintéticos reutilizables a partir de configuraciones YAML

**Versión**: 5.0 (MEJORADO con detección automática de datasets especiales)

**Aplicabilidad**: ✅ Agnóstico a dominio (retail, ecommerce, CRM, supply chain, etc.)

**Plataforma**: ⚠️ Microsoft Fabric (Lakehouse) — requiere `mssparkutils`

**Características**:
- 📋 Lee configuraciones desde YAML
- 🎲 Soporta múltiples generadores: sequence, faker, values, range, conditional, calculated, fk, weighted_values
- 📊 Genera parquets en Fabric/Lakehouse
- ✅ Validaciones automáticas
- 🔄 Ejecución batch o individual
- 🌍 Localizable
- ⭐ **NEW V5**: Detección automática de datasets especiales (product_changes)
- ⭐ **NEW V5**: Función generate_historical_changes para cambios históricos

---

## Cómo Usar

> ⚠️ **Prerequisito de plataforma**: Este notebook requiere Microsoft Fabric con un Lakehouse adjunto. Usa `mssparkutils` internamente para leer YAMLs y escribir parquets. No es compatible con Databricks ni Spark local en esta versión.

### Paso 1: Configurar Ambiente (Solo una vez)

Si usas **ambiente default** (sin environment.yml pre-configurado), ejecuta la celda de instalación:

```python
%pip install faker
%pip install faker-commerce
# ⚠️ Reinicia kernel automáticamente
```

**Si ya tienes environment con pandas, pyspark**: No necesitas instalar nada.

### Paso 2: Ejecutar Este Notebook

1. Ve a **Celda 2**: Ajusta `FAKER_LOCALE` si lo necesitas (default: es_ES)
2. Ve a **Celda 8**: Modifica `YAML_FILES` con tus archivos de configuración
3. Ejecuta todas las celdas en orden
4. Monitorea el resumen final

### Paso 3: Acceder a los Datos

Los parquets se guardan en:
```
Files/Data/{categoria}/{dataset_name}.parquet
```

Ej: `Files/Data/Stores/stores_base.parquet`

---

## Configuración YAML

### Datasets Normales (Con `rows`)

```yaml
dataset:
  name: my_dataset
  description: "..."
  rows: 1000                # Número de filas
  output: "Files/Data/MyData"
  columns:
    - name: id
      type: int
      sequence: {start: 1, step: 1}
    # ... más columnas
```

### Datasets Especiales (Con `source_parquet`) - **NEW V5**

```yaml
dataset:
  name: product_changes
  description: "Cambios históricos de productos"
  source_parquet: "Files/Data/Products/products_base.parquet"  # ← Special
  output: "Files/Data/Products"
  change_generation:
    change_ratio: 0.15  # 15% de productos
    num_changes_per_product: [1, 3]  # 1-3 cambios c/u
  columns:
    - name: ProductKey
      type: int
      from_source: true
    # ... más columnas
```

**Generadores Disponibles**:
| Generador | Descripción | Ejemplo |
|-----------|-------------|----------|
| `sequence` | Secuencial | start: 1, step: 1 |
| `faker` | Fake data (Faker) | name, email, phone_number |
| `values` | Lista fija | ["A", "B", "C"] |
| `range` | Rango numérico o fecha | [1, 100] o ["2020-01-01", "2025-12-31"] |
| `weighted_values` | Valores con pesos | [{value: "A", weight: 0.6}, ...] |
| `conditional` | Dependiente de otra columna | based_on: Category, mappings: {...} |
| `calculated` | Fórmula Python | formula: "ListPrice * 0.9" |
| `fk` | Foreign Key (lookup) | source: parquet_path, column: key_col |
| `template` | Plantilla con placeholders | "SKU-{sequence:05d}" |

---

## Arquitectura V5

```
YAML Config
    ↓
load_yaml_config()
    ↓
run_batch_generation() [DECISION POINT - V5 NEW]
    ├─ Si tiene 'source_parquet' → generate_historical_changes()
    └─ Si tiene 'rows' → generate_dataset()
    ├─ generate_column_value()
    ├─ Crear Pandas DataFrame
    └─ Convertir a PySpark DataFrame
    ↓
validate_dataset()
    ↓
write_parquet()
    ↓
Parquet en Lakehouse
```

---

## Parámetros Configurables (Celda 2)

| Parámetro | Tipo | Default | Propósito |
|-----------|------|---------|----------|
| `FAKER_LOCALE` | str | "es_ES" | Idioma para Faker (es_ES, en_US, fr_FR, etc.) |
| `RANDOM_SEED` | int\|None | 42 | Seed para reproducibilidad. `None` = aleatorio cada ejecución |
| `BATCH_SIZE` | int | 10000 | Filas por lote en generación |
| `BASE_OUTPUT_PATH` | str | "Files" | Ruta base del Lakehouse donde se escriben los parquets |
| `YAML_FILES` | list | — | Lista de YAMLs a generar, en orden de dependencias |

---

## Validaciones Automáticas

Al finalizar cada dataset:
1. ✅ **Número de filas**: Verifica que coincida con config (para datasets normales)
2. ✅ **Cambios históricos**: Verifica que sea > 0 (para product_changes)
3. ✅ **Secuencias únicas**: Valida que columnas 'sequence' no tengan duplicados
4. ✅ **Tipos de datos**: Convierte automáticamente según YAML
5. ✅ **Parquet escrito**: Verifica integridad del archivo

---

## Para Contribuyentes Open-Source

✅ **Genérico**: Sin referencias a dominio específico — adaptar vía YAML
✅ **Documentado**: Cada función tiene docstring
✅ **Reutilizable**: Solo personalizar YAML + Celda 2
✅ **Testeable**: Validaciones incluidas
✅ **Extensible**: Agregar nuevos tipos de datasets es trivial

**Mejoras futuras**:
- [ ] Soportar más tipos (JSON, array, struct)
- [ ] Paralelización de batch
- [ ] Profiling de performance
- [ ] Integración con Great Expectations
- [ ] Soporte para SCD Type 2 automático

---

## Ejemplos de Uso

### Ejemplo 1: Generar Dimensión de Tiendas

```yaml
# stores.yml
dataset:
  name: stores
  rows: 100
  columns:
    - name: StoreKey
      type: int
      sequence: {start: 1, step: 1}
    - name: StoreName
      type: string
      faker: company
    - name: City
      type: string
      faker: city
```

### Ejemplo 2: Generar Cambios Históricos de Productos (NEW V5)

```yaml
# product_changes.yml
dataset:
  name: product_changes
  source_parquet: "Files/Data/Products/products_base.parquet"
  change_generation:
    change_ratio: 0.15
    num_changes_per_product: [1, 3]
  columns:
    - name: ProductKey
      type: int
      from_source: true
    - name: ChangeSequence
      type: int
```

---

**Versión**: 5.0 (MEJORADO)
**Última actualización**: 2025-12-17
**Autor**: Javier Loria
**Licencia**: MIT — ver LICENSE


In [ ]:
# DESCOMENTAR si necesitas instalar (ambiente default, sin env.yml)
# %pip install faker
# %pip install faker-commerce
# print("✅ Librerías instaladas. Kernel reiniciado.")

In [ ]:
# ============================================================================
# IMPORTACIONES
# ============================================================================

import yaml
from faker import Faker
import random
import math
from datetime import datetime, timedelta, date
from decimal import Decimal
import pandas as pd
import traceback
from tqdm import tqdm  # Para progreso

from pyspark.sql import Row, SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

# ============================================================================
# CONFIGURACIÓN GLOBAL (Personalizar aquí)
# ============================================================================

FAKER_LOCALE = "es_ES"          # Idioma: es_ES, en_US, fr_FR, de_DE, it_IT, etc.
RANDOM_SEED = 42                # Reproducible: misma seed = mismos datos. None = aleatorio cada vez.
BATCH_SIZE = 10000              # Filas por lote en generación
BASE_OUTPUT_PATH = "Files"      # Ruta base de outputs (Fabric default: "Files")

# Lista de YAMLs a generar (en orden de dependencias)
YAML_FILES = [
    "Files/config/employees.yml",
    "Files/config/products_catalog.yml",
    "Files/config/support_tickets.yml",
    "Files/config/iot_readings.yml",
]

# Inicializar Faker
fake = Faker(FAKER_LOCALE)
Faker.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Obtener SparkSession
spark = SparkSession.builder.appName("GeneradorDatos").getOrCreate()

# Custom providers (comercio)
try:
    from faker_commerce import Provider as CommerceProvider
    fake.add_provider(CommerceProvider)
    print("✅ Faker Commerce Provider cargado")
except ImportError:
    print("⚠️  faker_commerce no instalado")
    print("   Para usarlo: %pip install faker-commerce")

print(f"🎲 Generador de Datos Sintéticos v5.0")
print(f"🎯 Seed: {RANDOM_SEED} | Locale: {FAKER_LOCALE}")
print(f"{'='*80}\n")

In [ ]:
## Cargar Configuración YAML
def load_yaml_config(yaml_path):
    """Carga configuración desde YAML del lakehouse."""
    if not yaml_path:
        raise ValueError("yaml_path no puede ser None o vacío")
    try:
        text = mssparkutils.fs.head(yaml_path, 10_000_000)
        config = yaml.safe_load(text)
    except Exception as e:
        raise FileNotFoundError(f"No se pudo leer {yaml_path}: {str(e)}")

    if config is None:
        raise ValueError(f"YAML vacío o inválido: {yaml_path}")
    if 'dataset' not in config:
        raise ValueError("YAML debe contener sección 'dataset'")

    return config['dataset']

print("✅ Función load_yaml_config definida")

In [ ]:
# ============================================================================
# FUNCIONES GENERADORAS BÁSICAS
# ============================================================================

def generate_sequence_value(col_config, row_index):
    """Genera valores secuenciales."""
    seq = col_config.get('sequence', {})
    start = seq.get('start', 1)
    step = seq.get('step', 1)
    return start + (row_index * step)

def generate_faker_value(col_config):
    """Genera valores con Faker (soporta dot notation)."""
    faker_method = col_config.get('faker')

    if not faker_method:
        raise ValueError(f"Columna '{col_config['name']}' sin 'faker' definido")

    # Manejar dot notation (ej: "commerce.product_name")
    if '.' in faker_method:
        parts = faker_method.split('.')
        obj = fake
        for part in parts:
            if not hasattr(obj, part):
                raise ValueError(f"Método Faker inválido: '{faker_method}'")
            obj = getattr(obj, part)
        return obj() if callable(obj) else obj
    else:
        if not hasattr(fake, faker_method):
            raise ValueError(f"Método Faker inválido: '{faker_method}'")
        method = getattr(fake, faker_method)
        return method() if callable(method) else method

def generate_choice_value(col_config):
    """Genera valores de lista fija."""
    values = col_config.get('values', [])
    if not values:
        raise ValueError(f"Columna '{col_config['name']}' sin valores")
    return random.choice(values)

def generate_range_value(col_config, col_type):
    """Genera valores en rango."""
    range_vals = col_config.get('range', [])
    if len(range_vals) != 2:
        raise ValueError(f"Rango debe tener 2 valores: {range_vals}")

    if col_type == 'date':
        start_date = datetime.strptime(range_vals[0], '%Y-%m-%d').date()
        end_date = datetime.strptime(range_vals[1], '%Y-%m-%d').date()
        delta = (end_date - start_date).days
        return start_date + timedelta(days=random.randint(0, delta))
    elif col_type == 'decimal':
        return round(Decimal(str(random.uniform(float(range_vals[0]), float(range_vals[1])))), 2)
    elif col_type == 'int':
        return random.randint(int(range_vals[0]), int(range_vals[1]))
    else:
        raise ValueError(f"Tipo no soportado para range: {col_type}")

def generate_bool_value(col_config):
    """Genera booleanos según ratio."""
    true_ratio = col_config.get('true_ratio', 0.5)
    return random.random() < true_ratio

print("✅ Funciones generadoras básicas definidas")

In [ ]:
# ============================================================================
# FUNCIONES GENERADORAS AVANZADAS
# ============================================================================

def generate_fk_value(col_config, valid_keys_cache):
    """Genera FK desde parquet ya generado."""
    fk_config = col_config.get('fk', {})
    source_path = fk_config.get('source')
    source_column = fk_config.get('column')

    cache_key = f"{source_path}:{source_column}"
    if cache_key not in valid_keys_cache:
        df_source = spark.read.parquet(source_path)
        keys = [row[source_column] for row in df_source.select(source_column).distinct().collect()]
        if not keys:
            raise ValueError(f"FK source '{source_path}' (columna '{source_column}') tiene 0 registros")
        valid_keys_cache[cache_key] = keys

    return random.choice(valid_keys_cache[cache_key])

def generate_weighted_choice_value(col_config):
    """Genera valores con pesos."""
    weighted_vals = col_config.get('weighted_values', [])
    if not weighted_vals:
        raise ValueError(f"weighted_values no definido para {col_config['name']}")
    values = [item['value'] for item in weighted_vals]
    weights = [item['weight'] for item in weighted_vals]
    return random.choices(values, weights=weights, k=1)[0]

def generate_conditional_value(col_config, row_data):
    """Genera valores condicionales según otra columna."""
    cond_config = col_config.get('conditional', {})
    based_on = cond_config.get('based_on')
    mappings = cond_config.get('mappings', {})

    if based_on not in row_data:
        raise ValueError(f"Columna '{based_on}' requerida para condicional")

    parent_value = row_data[based_on]
    options = mappings.get(parent_value, [])

    if not options:
        raise ValueError(f"No hay opciones para {based_on}={parent_value}")

    return random.choice(options)

def generate_template_value(col_config, row_index):
    """Genera valores desde template (ej: SKU-{sequence:05d})."""
    template = col_config.get('template', '')
    if '{sequence:' in template:
        import re
        match = re.search(r'\{sequence:([^}]+)\}', template)
        if match:
            fmt = match.group(1)
            seq_val = generate_sequence_value(col_config, row_index)
            formatted = ("{:" + fmt + "}").format(seq_val)
            template = template.replace(f"{{sequence:{fmt}}}", formatted)
    return template

def generate_calculated_value(col_config, row_data):
    """Genera valores calculados (fórmulas Python)."""
    calc_config = col_config.get('calculated', {})
    formula = calc_config.get('formula', '')
    based_on = calc_config.get('based_on', [])

    if not formula:
        raise ValueError(f"Columna {col_config['name']} sin fórmula")

    if isinstance(based_on, str):
        based_on = [based_on]

    for dep in based_on:
        if dep not in row_data:
            raise ValueError(f"Dependencia '{dep}' no existe para {col_config['name']}")

    # SafeRandom para evitar problemas Decimal*float
    class SafeRandom:
        def uniform(self, a, b):
            return random.uniform(float(a), float(b))
        def randint(self, a, b):
            return random.randint(int(a), int(b))
        def choice(self, seq):
            return random.choice(seq)

    safe_context = {
        'random': SafeRandom(),
        'int': int,
        'float': float,
        'str': str,
        'Decimal': Decimal,
        'datetime': datetime,
        'date': date,
        'timedelta': timedelta,
    }
    safe_context.update(row_data)

    try:
        return eval(formula, {"__builtins__": {}}, safe_context)
    except Exception as e:
        raise ValueError(f"Error en fórmula '{formula}': {str(e)}")

print("✅ Funciones generadoras avanzadas definidas")

In [ ]:
# ============================================================================
# FUNCIÓN ESPECIAL: Generar Cambios Históricos (datasets con source_parquet)
# ============================================================================

def generate_historical_changes(config):
    """
    Genera cambios históricos para datasets con 'source_parquet'.
    Totalmente data-driven: lee 'from_source', 'variation' y 'change_probability'
    del YAML — sin hardcoding por nombre de columna.

    Parámetros YAML:
      - source_parquet: ruta al parquet base
      - change_generation.change_ratio: % de filas con cambios (ej: 0.15 = 15%)
      - change_generation.num_changes_per_product: rango [min, max] cambios por fila
      - columns[*].from_source: nombre del campo en el parquet fuente
      - columns[*].variation: [min_pct, max_pct] para variación numérica
      - columns[*].change_probability + faker: probabilidad de cambio con Faker
    """
    source_path = config.get('source_parquet')
    change_ratio = config.get('change_generation', {}).get('change_ratio', 0.15)
    num_changes_range = config.get('change_generation', {}).get('num_changes_per_product', [1, 3])

    print(f"\n🔄 Generando cambios históricos desde {source_path}...")
    print(f"   Cobertura: {change_ratio*100:.0f}% de registros")

    # Leer solo las columnas requeridas del parquet fuente (derivadas del YAML)
    df_products = spark.read.parquet(source_path)
    source_cols = list(set(
        col['from_source'] for col in config['columns']
        if 'from_source' in col and isinstance(col['from_source'], str)
    ))
    products = df_products.select(*source_cols).collect()

    # Seleccionar registros a cambiar
    num_to_change = max(1, int(len(products) * change_ratio))
    changing_products = random.sample(products, num_to_change)

    all_rows = []
    valid_keys_cache = {}
    row_counter = 0  # Contador global para generadores de secuencia

    from tqdm import tqdm
    for product in tqdm(changing_products, desc="Generando cambios"):
        num_changes = random.randint(num_changes_range[0], num_changes_range[1])

        for _ in range(num_changes):
            row_data = {}

            for col in config['columns']:
                col_name = col['name']

                if 'from_source' in col:
                    source_field = col['from_source']
                    if 'variation' in col:
                        # Variación numérica ±%
                        variation = col['variation']
                        base_value = float(product[source_field])
                        var_amount = random.uniform(variation[0], variation[1])
                        row_data[col_name] = round(base_value * (1 + var_amount), 2)
                    elif 'change_probability' in col:
                        # Cambio probabilístico con Faker
                        change_prob = col['change_probability']
                        if random.random() < change_prob:
                            row_data[col_name] = generate_faker_value(col)
                        else:
                            row_data[col_name] = product[source_field]
                    else:
                        # Copia directa del campo fuente
                        row_data[col_name] = product[source_field]
                else:
                    # Generador estándar (sequence, range, values, faker, etc.)
                    value = generate_column_value(col, row_counter, row_data, valid_keys_cache)
                    row_data[col_name] = value

            all_rows.append(row_data)
            row_counter += 1

    print(f"✓ Generados {len(all_rows)} cambios para {num_to_change} registros (~{len(all_rows)/max(1,num_to_change):.1f} cambios/registro)")

    # Crear DataFrame de Pandas
    print("\n📊 Creando DataFrame de Pandas...")
    df_pandas = pd.DataFrame(all_rows)

    # Convertir tipos
    for col in config['columns']:
        col_name = col['name']
        if col_name not in df_pandas.columns:
            continue
        col_type = col.get('type', 'string')

        if col_type == 'int':
            df_pandas[col_name] = pd.to_numeric(df_pandas[col_name], errors='coerce').astype('Int64')
        elif col_type == 'date':
            df_pandas[col_name] = pd.to_datetime(df_pandas[col_name])
        elif col_type == 'decimal':
            df_pandas[col_name] = pd.to_numeric(df_pandas[col_name], errors='coerce').astype('float64')

    print("⚡ Convirtiendo a PySpark...")
    df_spark = spark.createDataFrame(df_pandas)

    print(f"✓ Dataset: {df_spark.count():,} filas de cambios históricos")

    return df_spark

print("✅ Función generate_historical_changes definida (data-driven)")

In [ ]:
def generate_column_value(col_config, row_index, row_data, valid_keys_cache):
    """Genera valor para columna con prioridad."""
    col_type = col_config.get('type', 'string')
    col_name = col_config.get('name')

    try:
        # Nullable PRIMERO
        if 'nullable' in col_config:
            if random.random() < col_config.get('nullable', 0.0):
                return None

        # Prioridad de generadores
        if 'sequence' in col_config:
            return generate_sequence_value(col_config, row_index)
        elif 'template' in col_config:
            return generate_template_value(col_config, row_index)
        elif 'fk' in col_config:
            return generate_fk_value(col_config, valid_keys_cache)
        elif 'weighted_values' in col_config:
            return generate_weighted_choice_value(col_config)
        elif 'conditional' in col_config:
            return generate_conditional_value(col_config, row_data)
        elif 'faker' in col_config:
            return generate_faker_value(col_config)
        elif 'values' in col_config:
            return generate_choice_value(col_config)
        elif 'range' in col_config:
            return generate_range_value(col_config, col_type)
        elif col_type == 'bool':
            return generate_bool_value(col_config)
        elif 'calculated' in col_config:
            return generate_calculated_value(col_config, row_data)
        else:
            defaults = {'int': 0, 'string': '', 'date': datetime.now().date(), 
                       'bool': False, 'decimal': Decimal('0.00')}
            return defaults.get(col_type, None)
    except Exception as e:
        raise ValueError(f"Error en columna '{col_name}': {str(e)}")

def generate_dataset(config):
    """Genera dataset genérico desde YAML."""
    num_rows = config.get('rows', 0)
    columns = config['columns']

    if num_rows == 0:
        raise ValueError("'rows' debe ser > 0")

    print(f"\n🔄 Generando {num_rows:,} filas para '{config['name']}'...")

    all_rows = []
    valid_keys_cache = {}

    total_batches = (num_rows + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_num in tqdm(range(total_batches), desc="Lotes"):
        batch_start = batch_num * BATCH_SIZE
        batch_end = min(batch_start + BATCH_SIZE, num_rows)

        for row_idx in range(batch_start, batch_end):
            row_data = {}

            # Pass 1: Generar no-dependientes (sin conditional, calculated, ni fk)
            # Criterio: columnas con sequence, template, faker, values, range, weighted_values, true_ratio/bool
            for col in columns:
                # Skip si es condicional, calculado, o FK
                if 'conditional' not in col and 'calculated' not in col and 'fk' not in col:
                    value = generate_column_value(col, row_idx, row_data, valid_keys_cache)
                    row_data[col['name']] = value

            # Pass 2: Generar dependientes (condicionales, calculadas, FK)
            # Estos dependen de valores generados en Pass 1
            for col in columns:
                col_name = col['name']
                if col_name not in row_data:
                    value = generate_column_value(col, row_idx, row_data, valid_keys_cache)
                    row_data[col_name] = value

            all_rows.append(row_data)

    print("\n📊 Creando DataFrame de Pandas...")
    df_pandas = pd.DataFrame(all_rows)

    # Convertir tipos
    for col in columns:
        col_name = col['name']
        col_type = col.get('type', 'string')

        if col_name not in df_pandas.columns:
            continue

        if col_type == 'int':
            df_pandas[col_name] = df_pandas[col_name].astype('Int64')
        elif col_type == 'string':
            df_pandas[col_name] = df_pandas[col_name].astype(str)
        elif col_type == 'date':
            df_pandas[col_name] = pd.to_datetime(df_pandas[col_name])
        elif col_type == 'bool':
            df_pandas[col_name] = df_pandas[col_name].astype('boolean')
        elif col_type == 'decimal':
            df_pandas[col_name] = df_pandas[col_name].astype('float64')

    print("⚡ Convirtiendo a PySpark...")
    df_spark = spark.createDataFrame(df_pandas)

    del df_pandas, all_rows

    print(f"✓ Dataset: {df_spark.count():,} filas, {len(df_spark.columns)} columnas")

    return df_spark

print("✅ Función generate_dataset definida (con Pass 1/Pass 2 corregido)")

In [ ]:
def validate_dataset(df, config):
    """Valida dataset generado."""
    print("\n🔍 Validaciones...")

    validation_results = {'passed': True, 'errors': []}

    # Validar número de filas
    if 'rows' in config:
        expected = config['rows']
        actual = df.count()
        if actual == expected:
            print(f"   ✓ Filas: {actual:,}")
        else:
            msg = f"Filas: esperado {expected:,}, obtenido {actual:,}"
            print(f"   ✗ {msg}")
            validation_results['passed'] = False
            validation_results['errors'].append(msg)

    # Para datasets de cambios históricos, validar que sea > 0
    elif 'source_parquet' in config:
        actual = df.count()
        if actual > 0:
            print(f"   ✓ Cambios históricos: {actual:,} filas")
        else:
            msg = "No se generaron cambios históricos"
            print(f"   ✗ {msg}")
            validation_results['passed'] = False
            validation_results['errors'].append(msg)

    # Validar secuencias únicas
    for col_cfg in config.get('columns', []):
        if 'sequence' in col_cfg:
            col_name = col_cfg['name']
            total = df.count()
            distinct = df.select(col_name).distinct().count()
            if total == distinct:
                print(f"   ✓ {col_name}: únicos")
            else:
                msg = f"Columna '{col_name}': {total - distinct:,} duplicados"
                print(f"   ✗ {msg}")
                validation_results['passed'] = False
                validation_results['errors'].append(msg)

    return validation_results

def write_parquet(df, config):
    """Escribe DataFrame a Parquet."""
    output_path = config.get('output', f"Files/Data/{config['name']}")

    if not output_path.startswith('Files/'):
        output_path = f"Files/{output_path}"

    temp_folder = f"{output_path}/_temp_{config['name']}"
    final_file = f"{output_path}/{config['name']}.parquet"

    print(f"\n💾 Escribiendo: {final_file}")

    try:
        df.coalesce(1).write.mode("overwrite").option("compression", "snappy").parquet(temp_folder)

        temp_files = mssparkutils.fs.ls(temp_folder)
        parquet_files = [f for f in temp_files if f.name.endswith('.parquet')]
        if not parquet_files:
            raise ValueError(f"No se encontró archivo .parquet en {temp_folder}")
        parquet_file = parquet_files[0]
        temp_file_path = f"{temp_folder}/{parquet_file.name}"

        try:
            mssparkutils.fs.rm(final_file)
        except:
            pass

        mssparkutils.fs.cp(temp_file_path, final_file)

        verify_df = spark.read.parquet(final_file)
        print(f"   ✓ Parquet: {verify_df.count():,} filas")

        return final_file
    except Exception as e:
        print(f"   ✗ Error: {str(e)}")
        raise
    finally:
        try:
            mssparkutils.fs.rm(temp_folder, recurse=True)
        except:
            pass

def run_batch_generation(yaml_files):
    """Ejecuta generación batch de múltiples YAMLs.

    Detecta automáticamente tipo de dataset:
      - Si tiene 'source_parquet' → genera cambios históricos
      - Si tiene 'rows' → genera dataset estándar
    """
    print(f"\n{'='*80}")
    print(f"🚀 BATCH: {len(yaml_files)} datasets")
    print(f"{'='*80}\n")

    results = {}

    for idx, yaml_file in enumerate(yaml_files, 1):
        print(f"\n{'─'*80}")
        print(f"[{idx}/{len(yaml_files)}] {yaml_file}")
        print(f"{'─'*80}")

        dataset_name = yaml_file  # Fallback si load_yaml_config falla
        try:
            config = load_yaml_config(yaml_file)
            dataset_name = config['name']

            if 'source_parquet' in config:
                print(f"📝 Tipo: HISTÓRICO (source_parquet: {config.get('source_parquet')})")
                df = generate_historical_changes(config)
            elif 'rows' in config:
                print(f"📝 Tipo: ESTÁNDAR (rows: {config.get('rows')})")
                df = generate_dataset(config)
            else:
                raise ValueError("YAML debe tener 'rows' o 'source_parquet'")

            validation_result = validate_dataset(df, config)
            if not validation_result['passed']:
                errors = '; '.join(validation_result.get('errors', ['ver salida arriba']))
                raise ValueError(f"Validación fallida para '{dataset_name}': {errors}")

            parquet_path = write_parquet(df, config)

            results[dataset_name] = {'status': 'OK', 'rows': df.count()}
            print(f"\n✅ Completado")

        except Exception as e:
            print(f"\n❌ ERROR: {str(e)}")
            traceback.print_exc()
            results[dataset_name] = {'status': 'ERROR', 'error': str(e)}

    # Resumen
    print(f"\n{'='*80}")
    print(f"📊 RESUMEN")
    print(f"{'='*80}")

    ok_count = 0
    error_count = 0

    for name, result in results.items():
        if result['status'] == 'OK':
            print(f"✅ {name}: {result['rows']:,} filas")
            ok_count += 1
        else:
            print(f"❌ {name}: {result['error']}")
            error_count += 1

    print(f"\n📈 Total: {ok_count} ✅ | {error_count} ❌")

    return results

print("✅ Funciones de validación y escritura definidas")

In [ ]:
# ============================================================================
# EJECUTAR
# ============================================================================

print("\n🎯 Iniciando generación...\n")
results = run_batch_generation(YAML_FILES)
print("\n🎉 PROCESO COMPLETADO")